In [8]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [7]:
# pip install numpy

In [6]:
# pip install paddlepaddle

In [4]:
import torch, paddle, sys
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Paddle:", paddle.__version__, "Device:", paddle.get_device())

Python: 3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)]
Torch: 2.6.0+cu124 CUDA: True
Paddle: 3.2.1 Device: cpu


In [11]:
import paddle
import os

def inspect_paddle_weights_simple(paddle_path):
    """
    Inspect PaddlePaddle weights without running GPU operations.
    This works even if CUDNN is not properly configured.
    """
    print("="*60)
    print(f"Inspecting: {paddle_path}")
    print("="*60)
    
    if not os.path.exists(paddle_path):
        print(f"✗ File not found: {paddle_path}")
        return None
    
    try:
        # Load weights (this doesn't require GPU/CUDNN)
        print("\nLoading weights...")
        state_dict = paddle.load(paddle_path)
        
        print(f"✓ Successfully loaded!")
        print(f"✓ Type: {type(state_dict)}")
        
        # Analyze structure
        if isinstance(state_dict, dict):
            print(f"✓ Number of top-level keys: {len(state_dict)}")
            print(f"\nTop-level keys: {list(state_dict.keys())}")
            
            # Try to find the actual model weights
            model_dict = state_dict
            
            # Common locations for model weights in Paddle checkpoints
            for key in ['model', 'state_dict', 'model_state_dict']:
                if key in state_dict:
                    print(f"\nFound '{key}' in checkpoint, using that as model dict")
                    model_dict = state_dict[key]
                    break
            
            # Display structure
            print(f"\nModel dict has {len(model_dict)} keys")
            print("\nFirst 20 keys with shapes:")
            for i, (key, value) in enumerate(list(model_dict.items())[:20]):
                if hasattr(value, 'shape'):
                    print(f"  {i+1}. {key}: {value.shape}")
                else:
                    print(f"  {i+1}. {key}: {type(value)}")
            
            if len(model_dict) > 20:
                print(f"  ... and {len(model_dict) - 20} more keys")
            
            # Search for class-related layers
            print("\n" + "="*60)
            print("Searching for Class-Related Layers")
            print("="*60)
            
            num_classes = None
            class_related = []
            
            for key, value in model_dict.items():
                if hasattr(value, 'shape'):
                    # Look for class/score/embed layers
                    if any(x in key.lower() for x in ['class_embed', 'cls', 'score', 'num_classes']):
                        # Get actual shape by calling the method
                        shape = value.shape() if callable(value.shape) else value.shape
                        class_related.append((key, shape))
            
            if class_related:
                print(f"\nFound {len(class_related)} class-related layers:")
                for key, shape in class_related:
                    # Call shape() method to get actual shape
                    actual_shape = shape() if callable(shape) else shape
                    print(f"  {key}: {actual_shape}")
                    
                    # Try to infer number of classes
                    if 'class_embed' in key.lower():
                        if 'bias' in key and len(actual_shape) == 1:
                            num_classes = actual_shape[0]
                            print(f"    → Found {num_classes} classes from bias!")
                        elif 'weight' in key and len(actual_shape) == 2:
                            potential = actual_shape[0]
                            print(f"    → Output dimension: {potential}")
                            if num_classes is None:
                                num_classes = potential
            else:
                print("\n✗ No obvious class-related layers found")
                print("Searching for decoder layers...")
                
                decoder_layers = []
                for key in model_dict.keys():
                    if 'decoder' in key.lower() or 'dec' in key.lower():
                        decoder_layers.append(key)
                
                if decoder_layers:
                    print(f"\nFound {len(decoder_layers)} decoder-related layers")
                    print("Sample decoder layers:")
                    for key in decoder_layers[:5]:
                        print(f"  {key}")
            
            # Final result
            print("\n" + "="*60)
            print("RESULT")
            print("="*60)
            
            if num_classes:
                print(f"✓✓✓ Number of classes: {num_classes}")
                
                if num_classes == 365:
                    print("✓✓✓ THIS IS AN OBJECTS365 MODEL (365 classes)!")
                    print("    You can use this for evaluation on Open Images v7!")
                elif num_classes == 80:
                    print("⚠ This is a COCO-finetuned model (80 classes)")
                    print("  NOT the Objects365-only version you need")
                else:
                    print(f"? Unknown dataset ({num_classes} classes)")
            else:
                print("✗ Could not determine number of classes")
                print("  The model might use a different architecture")
                print("  or the weights are organized differently")
            
            return num_classes
            
        else:
            print(f"✗ State dict is not a dictionary: {type(state_dict)}")
            return None
            
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return None

paddle_path = r"C:\Users\User\Downloads\rtdetr_r101vd_1x_objects365.pdparams"

# Check if file exists before trying
if os.path.exists(paddle_path):
    num_classes = inspect_paddle_weights_simple(paddle_path)
else:
    print(f"\nPlease update 'paddle_path' to point to your downloaded .pdparams file")
    print("\nTo download Objects365-only weights:")
    print("1. Go to: https://github.com/lyuwenyu/RT-DETR/tree/main/rtdetr_paddle")
    print("2. Find 'Model Zoo on Objects365' table")
    print("3. Click 'download' for '1x Objects365' models (NOT 'COCO + Objects365')")
    print("   - RT-DETR-R18 1x Objects365 (22.9 mAP)")
    print("   - RT-DETR-R50 1x Objects365 (35.1 mAP)")
    print("   - RT-DETR-R101 1x Objects365 (36.8 mAP)")

Inspecting: C:\Users\User\Downloads\rtdetr_r101vd_1x_objects365.pdparams

Loading weights...
✓ Successfully loaded!
✓ Type: <class 'dict'>
✓ Number of top-level keys: 949

Top-level keys: ['backbone.conv1.conv1_1.conv.weight', 'backbone.conv1.conv1_1.norm.weight', 'backbone.conv1.conv1_1.norm.bias', 'backbone.conv1.conv1_1.norm._mean', 'backbone.conv1.conv1_1.norm._variance', 'backbone.conv1.conv1_2.conv.weight', 'backbone.conv1.conv1_2.norm.weight', 'backbone.conv1.conv1_2.norm.bias', 'backbone.conv1.conv1_2.norm._mean', 'backbone.conv1.conv1_2.norm._variance', 'backbone.conv1.conv1_3.conv.weight', 'backbone.conv1.conv1_3.norm.weight', 'backbone.conv1.conv1_3.norm.bias', 'backbone.conv1.conv1_3.norm._mean', 'backbone.conv1.conv1_3.norm._variance', 'backbone.res2.res2a.branch2a.conv.weight', 'backbone.res2.res2a.branch2a.norm.weight', 'backbone.res2.res2a.branch2a.norm.bias', 'backbone.res2.res2a.branch2a.norm._mean', 'backbone.res2.res2a.branch2a.norm._variance', 'backbone.res2.res2a.

In [ ]:
"""
Convert PaddlePaddle RT-DETR weights to PyTorch format.
This script converts Objects365-trained RT-DETR weights from PaddlePaddle (.pdparams)
to PyTorch (.pth) format.

✅ Works with:
   - torch (GPU or CPU)
   - paddlepaddle (CPU version)
"""

import paddle
import torch
import numpy as np
import os

def convert_paddle_to_pytorch(paddle_path, pytorch_path, num_classes=365):
    """
    Convert PaddlePaddle RT-DETR weights to PyTorch format.
    
    Args:
        paddle_path: Path to .pdparams file
        pytorch_path: Output path for .pth file
        num_classes: Number of classes (365 for Objects365)
    """
    print("=" * 70)
    print("Converting Paddle RT-DETR → PyTorch")
    print("=" * 70)
    print(f"Input : {paddle_path}")
    print(f"Output: {pytorch_path}")
    print(f"Classes: {num_classes}")
    print()

    # ✅ Force Paddle to CPU
    paddle.set_device("cpu")
    print("✓ Paddle device set to:", paddle.get_device())

    # Load Paddle weights
    print("\nLoading Paddle weights...")
    paddle_state_dict = paddle.load(paddle_path)
    print(f"✓ Loaded {len(paddle_state_dict)} parameters")

    # Convert to PyTorch
    print("\nConverting to PyTorch format...")
    pytorch_state_dict = {}

    converted, skipped = 0, 0

    for key, value in paddle_state_dict.items():
        try:
            # Skip batch norm running stats
            if '_mean' in key or '_variance' in key:
                skipped += 1
                continue

            # Convert Paddle tensor → numpy → torch
            numpy_value = value.numpy() if hasattr(value, 'numpy') else np.array(value)
            torch_value = torch.from_numpy(numpy_value)

            pytorch_state_dict[key] = torch_value
            converted += 1

            if converted % 100 == 0:
                print(f"  Converted {converted} parameters...")

        except Exception as e:
            print(f"⚠️  Warning: Could not convert {key}: {e}")
            skipped += 1

    print(f"\n✓ Converted {converted} parameters")
    print(f"✓ Skipped {skipped} parameters (BN stats or incompatible shapes)")

    # Create PyTorch checkpoint
    checkpoint = {
        'model': pytorch_state_dict,
        'num_classes': num_classes,
        'source': 'paddle_rtdetr_objects365',
        'notes': 'Converted from PaddlePaddle RT-DETR Objects365 weights'
    }

    # Save PyTorch weights
    print(f"\nSaving to {pytorch_path} ...")
    torch.save(checkpoint, pytorch_path)
    print("✓ Saved successfully!")

    # Verify saved file
    print("\nVerifying saved weights...")
    loaded = torch.load(pytorch_path, map_location='cpu')
    print("✓ Verification successful!")
    print(f"  - Model keys: {len(loaded['model'])}")
    print(f"  - Num classes: {loaded.get('num_classes', 'unknown')}")

    return pytorch_state_dict


def quick_verify_conversion(pytorch_path):
    """Quick verification of converted PyTorch weights."""
    print("\n" + "=" * 70)
    print("Quick Verification")
    print("=" * 70)

    checkpoint = torch.load(pytorch_path, map_location='cpu')
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

    print(f"✓ Total parameters: {len(state_dict)}")
    print(f"✓ Num classes: {checkpoint.get('num_classes', 'Not specified')}")

    print("\nClass-related layers:")
    for key, value in state_dict.items():
        if 'class_embed' in key.lower() or 'score_head' in key.lower():
            print(f"  {key}: {tuple(value.shape)}")

    print("\n✓ Conversion appears successful!")

# 🛠️ Update these paths to match your system
paddle_path = r"C:\Users\User\Downloads\rtdetr_r101vd_1x_objects365.pdparams"
pytorch_path = r"C:\Users\User\Downloads\rtdetr_r101vd_objects365_pytorch.pt"

# Check file exists
if not os.path.exists(paddle_path):
    print(f"❌ Error: Paddle weights not found at {paddle_path}")
    print("Please update the 'paddle_path' variable above.")
else:
    pytorch_state_dict = convert_paddle_to_pytorch(
        paddle_path=paddle_path,
        pytorch_path=pytorch_path,
        num_classes=365
    )
    quick_verify_conversion(pytorch_path)

    print("\n" + "=" * 70)
    print("Next Steps:")
    print("=" * 70)
    print(f"1️⃣ PyTorch weights saved at: {pytorch_path}")
    print("\n2️⃣ To use with Ultralytics or other PyTorch projects:")
    print("   - Rename to .pt if needed")
    print("   - Or modify load script to accept .pth files")
    print("\n3️⃣ These weights have 365 classes (Objects365)")
    print("   → Gives ~200+ overlapping classes with OpenImages v7")
    print("=" * 70)

Converting Paddle RT-DETR → PyTorch
Input : C:\Users\User\Downloads\rtdetr_r101vd_1x_objects365.pdparams
Output: C:\Users\User\Downloads\rtdetr_r101vd_objects365_pytorch.pth
Classes: 365

✓ Paddle device set to: cpu

Loading Paddle weights...
✓ Loaded 949 parameters

Converting to PyTorch format...
  Converted 100 parameters...
  Converted 200 parameters...
  Converted 300 parameters...
  Converted 400 parameters...
  Converted 500 parameters...
  Converted 600 parameters...

✓ Converted 653 parameters
✓ Skipped 296 parameters (BN stats or incompatible shapes)

Saving to C:\Users\User\Downloads\rtdetr_r101vd_objects365_pytorch.pth ...
✓ Saved successfully!

Verifying saved weights...
✓ Verification successful!
  - Model keys: 653
  - Num classes: 365

Quick Verification
✓ Total parameters: 653
✓ Num classes: 365

Class-related layers:
  transformer.denoising_class_embed.weight: (365, 256)
  transformer.enc_score_head.weight: (256, 365)
  transformer.enc_score_head.bias: (365,)
  transf

In [10]:
import torch

ckpt = torch.load(r"C:\Users\User\Downloads\rtdetr_r101vd_objects365_pytorch.pt", map_location="cpu")
state_dict = ckpt['model']
